### read csv file in dataframe

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalesce.partitions.enabled", False)
spark.conf.set("spark.sql.autoBroadCastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", True)
spark.conf.set("spark.sql.shuffle.partitions", 8)
spark.conf.set("spark.default.parallelism", 8)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType

schema=StructType([StructField('src_region', StringType(), True), StructField('src_country', StringType(), True), StructField('src_item_type', StringType(), True), StructField('src_sales_channel', StringType(), True), StructField('src_order_Priority', StringType(), True), StructField('src_order_dt', StringType(), True), StructField('src_order_id', IntegerType(), True), StructField('src_ship_date',StringType(), True), StructField('src_units_sold', IntegerType(), True), StructField('src_unit_price', DoubleType(), True), StructField('src_unit_cost', DoubleType(), True), StructField('src_total_revenue', DoubleType(), True), StructField('src_total_cost', DoubleType(), True), StructField('total_profit', DoubleType(), True)])

df = spark.read.format("csv") \
    .option("header","true") \
    .option("sep", ",") \
    .option("nullValue", "NULL") \
    .option("encoding", "UTF-8") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("multiLine", "true") \
    .option("ignoreLeadingWhiteSpace", "true") \
    .option("ignoreTrailingWhiteSpace", "true") \
    .schema(schema) \
    .load("/mnt/adls/bronze/lob1/source1/rewrite_csv")  # Path to CSV file (S3, HDFS, Azure, or local)


In [0]:
from pyspark.sql.functions import col  ,concat_ws,hash , lpad , date_format
df_new=df.withColumn('src_order_dt',lpad(col('src_order_dt'),10,"0")).drop(col("src_ship_date"))
all_columns_csv=df_new.columns
df_csv_hash=df_new.withColumn('hash_id_csv',hash(concat_ws("||", *all_columns_csv)))
df_par=spark.read.parquet("/mnt/adls/bronze/lob1/source1/parquet/*").drop(col("Ship Date"))
all_columns_par=df_par.columns
df_par_hash=df_par.drop('Ship Date').withColumn("Order Date",date_format(col('Order date'),"MM/dd/yyyy")).withColumn('hash_id_par',hash(concat_ws("||", *all_columns_par)))

df_join=df_csv_hash.join(df_par_hash, col('src_order_id')==col('Order ID'), "fullouter")
                                    

In [0]:
df_join.write.format("parquet").mode("overwrite").save('/mnt/adls/bronze/lob1/join_result/temp_2025_06_02')

In [0]:
df_join_read=spark.read.parquet("/mnt/adls/bronze/lob1/join_result/temp_2025_06_02")
deletes = df_join_read.filter(col("hash_id_csv").isNull())
inserts = df_join_read.filter(col("hash_id_par").isNull())
updates= df_join_read.filter( (col("hash_id_csv").isNotNull()) & (col("hash_id_par").isNotNull()) & (col("hash_id_csv")!=col("hash_id_par")) )
nochange= df_join_read.filter( (col("hash_id_csv").isNotNull()) & (col("hash_id_par").isNotNull()) & (col("hash_id_csv")==col("hash_id_par")) )

In [0]:
dbutils.fs.ls("/mnt/adls/bronze/lob1/source1/temp_2025_06_02/")